# ORBIT approval wait/resume demo
Notebook workbench surface for approval-required dummy runs.

In [10]:
from orbit.store import create_default_store
from orbit.runtime import OrbitCoordinator
from orbit.notebook import NotebookWorkbench, project_run
from orbit.settings import DEFAULT_WORKSPACE_ROOT

store = create_default_store()
coordinator = OrbitCoordinator(store, DEFAULT_WORKSPACE_ROOT)
workbench = NotebookWorkbench(coordinator)
run = coordinator.run_dummy_scenario(
    title='Notebook approval demo',
    description='Demonstrate wait/resume approval behavior',
    user_input='Create a governed file but require explicit approval first.',
    dummy_scenario='approval_required',
)
run.run_id, run.status

('run_76fe6f309c7f', <RunStatus.WAITING_FOR_APPROVAL: 'waiting_for_approval'>)

In [11]:
workbench.run_summary_dataframe(run.run_id)

,run_id,task_id,status,created_at,started_at,ended_at,current_step_id,result_summary,failure_reason
0,run_76fe6f309c7f,task_e4139d4f39ea,waiting_for_approval,2026-04-01 15:12:39.771934+00:00,2026-04-01 15:12:39.771912+00:00,None,step_87eddc3e82b6,None,None


In [3]:
workbench.steps_dataframe(run.run_id)

,step_id,run_id,index,step_type,status,started_at,ended_at
0,step_dce4e78e3b4e,run_1a7fe9d9de1c,1,context_assembly,completed,2026-04-01 12:02:19.816394+00:00,2026-04-01 12:02:19.818672+00:00
1,step_33d8e424f46d,run_1a7fe9d9de1c,2,model_generation,completed,2026-04-01 12:02:19.819794+00:00,2026-04-01 12:02:19.821427+00:00
2,step_d8c283b32f41,run_1a7fe9d9de1c,3,tool_execution,running,2026-04-01 12:02:19.823091+00:00,NaT


In [4]:
workbench.events_dataframe(run.run_id)

,event_id,run_id,event_type,timestamp,step_id,severity,payload
0,evt_62a8a3cd4ab0,run_1a7fe9d9de1c,run_created,2026-04-01 12:02:19.814081+00:00,None,None,"{'task_id': 'task_8a5baab5d772', 'session_key'..."
1,evt_a9aec85ba62d,run_1a7fe9d9de1c,run_started,2026-04-01 12:02:19.814661+00:00,None,None,{'user_input': 'Create a governed file but req...
2,evt_3bb4d0d2a1e7,run_1a7fe9d9de1c,step_started,2026-04-01 12:02:19.818132+00:00,step_dce4e78e3b4e,None,{'step_type': 'context_assembly'}
3,evt_86a53ca8a0ef,run_1a7fe9d9de1c,step_completed,2026-04-01 12:02:19.819328+00:00,step_dce4e78e3b4e,None,{'step_type': 'context_assembly'}
4,evt_9239fba2dba5,run_1a7fe9d9de1c,step_started,2026-04-01 12:02:19.820896+00:00,step_33d8e424f46d,None,{'step_type': 'model_generation'}
5,evt_c1bbbaeb62f0,run_1a7fe9d9de1c,step_completed,2026-04-01 12:02:19.821968+00:00,step_33d8e424f46d,None,{'step_type': 'model_generation'}
6,evt_1f1a2dd89915,run_1a7fe9d9de1c,dummy_plan_selected,2026-04-01 12:02:19.822520+00:00,None,None,{'scenario': 'approval_required'}
7,evt_bd5332788c6c,run_1a7fe9d9de1c,step_started,2026-04-01 12:02:19.824436+00:00,step_d8c283b32f41,None,{'step_type': 'tool_execution'}
8,evt_be5cb0464598,run_1a7fe9d9de1c,tool_invocation_requested,2026-04-01 12:02:19.825787+00:00,step_d8c283b32f41,None,"{'tool_name': 'write_file', 'scenario': 'appro..."
9,evt_6a54cf20dabb,run_1a7fe9d9de1c,approval_requested,2026-04-01 12:02:19.827780+00:00,step_d8c283b32f41,None,{'approval_request_id': 'approval_c51bac752d58'}


In [5]:
workbench.approval_requests_for_run_dataframe(run.run_id)

,approval_request_id,run_id,step_id,target_type,target_id,risk_level,status,created_at,reason
0,approval_c51bac752d58,run_1a7fe9d9de1c,step_d8c283b32f41,tool_invocation,tool_ab017a087922,review_required,open,2026-04-01 12:02:19.826433+00:00,write_file is side-effecting in dummy scenario...


In [6]:
approval_df = workbench.approval_requests_for_run_dataframe(run.run_id)
approval_id = approval_df.iloc[0]['approval_request_id']
resolved_run = workbench.approve(approval_id, 'approved from notebook workbench helper')
resolved_run.run_id, resolved_run.status

('run_1a7fe9d9de1c', <RunStatus.COMPLETED: 'completed'>)

In [7]:
workbench.run_summary_dataframe(run.run_id)

,run_id,task_id,status,created_at,started_at,ended_at,current_step_id,result_summary,failure_reason
0,run_1a7fe9d9de1c,task_8a5baab5d772,completed,2026-04-01 12:02:19.813361+00:00,2026-04-01 12:02:19.813357+00:00,2026-04-01 12:02:57.328867+00:00,step_c0bc072e4efb,Dummy runtime completed after approval-mediate...,None


In [8]:
workbench.events_dataframe(run.run_id)

,event_id,run_id,event_type,timestamp,step_id,severity,payload
0,evt_62a8a3cd4ab0,run_1a7fe9d9de1c,run_created,2026-04-01 12:02:19.814081+00:00,None,None,"{'task_id': 'task_8a5baab5d772', 'session_key'..."
1,evt_a9aec85ba62d,run_1a7fe9d9de1c,run_started,2026-04-01 12:02:19.814661+00:00,None,None,{'user_input': 'Create a governed file but req...
2,evt_3bb4d0d2a1e7,run_1a7fe9d9de1c,step_started,2026-04-01 12:02:19.818132+00:00,step_dce4e78e3b4e,None,{'step_type': 'context_assembly'}
3,evt_86a53ca8a0ef,run_1a7fe9d9de1c,step_completed,2026-04-01 12:02:19.819328+00:00,step_dce4e78e3b4e,None,{'step_type': 'context_assembly'}
4,evt_9239fba2dba5,run_1a7fe9d9de1c,step_started,2026-04-01 12:02:19.820896+00:00,step_33d8e424f46d,None,{'step_type': 'model_generation'}
5,evt_c1bbbaeb62f0,run_1a7fe9d9de1c,step_completed,2026-04-01 12:02:19.821968+00:00,step_33d8e424f46d,None,{'step_type': 'model_generation'}
6,evt_1f1a2dd89915,run_1a7fe9d9de1c,dummy_plan_selected,2026-04-01 12:02:19.822520+00:00,None,None,{'scenario': 'approval_required'}
7,evt_bd5332788c6c,run_1a7fe9d9de1c,step_started,2026-04-01 12:02:19.824436+00:00,step_d8c283b32f41,None,{'step_type': 'tool_execution'}
8,evt_be5cb0464598,run_1a7fe9d9de1c,tool_invocation_requested,2026-04-01 12:02:19.825787+00:00,step_d8c283b32f41,None,"{'tool_name': 'write_file', 'scenario': 'appro..."
9,evt_6a54cf20dabb,run_1a7fe9d9de1c,approval_requested,2026-04-01 12:02:19.827780+00:00,step_d8c283b32f41,None,{'approval_request_id': 'approval_c51bac752d58'}


In [9]:
project_run(store, run.run_id)

{'run': {'run_id': 'run_1a7fe9d9de1c',
  'task_id': 'task_8a5baab5d772',
  'status': 'completed',
  'created_at': '2026-04-01T12:02:19.813361Z',
  'started_at': '2026-04-01T12:02:19.813357Z',
  'ended_at': '2026-04-01T12:02:57.328867Z',
  'current_step_id': 'step_c0bc072e4efb',
  'result_summary': 'Dummy runtime completed after approval-mediated execution.',
  'failure_reason': None},
 'steps': [{'step_id': 'step_dce4e78e3b4e',
   'run_id': 'run_1a7fe9d9de1c',
   'step_type': 'context_assembly',
   'status': 'completed',
   'index': 1,
   'started_at': '2026-04-01T12:02:19.816394Z',
   'ended_at': '2026-04-01T12:02:19.818672Z'},
  {'step_id': 'step_33d8e424f46d',
   'run_id': 'run_1a7fe9d9de1c',
   'step_type': 'model_generation',
   'status': 'completed',
   'index': 2,
   'started_at': '2026-04-01T12:02:19.819794Z',
   'ended_at': '2026-04-01T12:02:19.821427Z'},
  {'step_id': 'step_d8c283b32f41',
   'run_id': 'run_1a7fe9d9de1c',
   'step_type': 'tool_execution',
   'status': 'complet